In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime

# 1. Load the MPG dataset
df = pd.read_csv('mpg.csv')
print("Original dataset shape:", df.shape)
print("\nOriginal dtypes:\n", df.dtypes)
print("\nFirst few rows:\n", df.head())

Original dataset shape: (398, 9)

Original dtypes:
 mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin           object
name             object
dtype: object

First few rows:
     mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  
3          70    usa              amc rebel sst  
4          70    usa         

In [4]:
# Check unique non-numeric values in horsepower
print("\nUnique values in horsepower (first 10):", df['horsepower'].unique()[:10])

# Coerce to numeric, setting invalid entries to NaN
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

print("\nAfter coercion, missing horsepower:", df['horsepower'].isna().sum())
print("New dtype:", df['horsepower'].dtype)


Unique values in horsepower (first 10): [130. 165. 150. 140. 198. 220. 215. 225. 190. 170.]

After coercion, missing horsepower: 6
New dtype: float64


In [5]:
# model_year is given as two-digit year (70 = 1970)
df['model_date'] = pd.to_datetime(df['model_year'], format='%y')
# Set to January 1 of that year
df['model_date'] = df['model_date'].apply(lambda dt: dt.replace(month=1, day=1))

# Extract year, month, quarter etc. if needed
df['model_year_full'] = df['model_date'].dt.year
print("\nSample of converted dates:")
print(df[['model_year', 'model_date', 'model_year_full']].head())


Sample of converted dates:
   model_year model_date  model_year_full
0          70 1970-01-01             1970
1          70 1970-01-01             1970
2          70 1970-01-01             1970
3          70 1970-01-01             1970
4          70 1970-01-01             1970


In [6]:
# First, normalise the name column (strip, lower, collapse spaces)
def normalize_string_column(series: pd.Series) -> pd.Series:
    """Apply strip, lower, and collapse internal spaces."""
    return series.str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

df['name_clean'] = normalize_string_column(df['name'])

# Extract manufacturer (first word)
df['manufacturer'] = df['name_clean'].str.split().str[0]

# Compare memory usage before and after converting to category
mem_before = df['manufacturer'].memory_usage(deep=True)
df['manufacturer'] = df['manufacturer'].astype('category')
mem_after = df['manufacturer'].memory_usage(deep=True)

saved_kb = (mem_before - mem_after) / 1024
print(f"\nMemory usage of 'manufacturer' as object: {mem_before / 1024:.2f} KB")
print(f"Memory usage as category: {mem_after / 1024:.2f} KB")
print(f"Memory saved: {saved_kb:.2f} KB ({saved_kb / mem_before * 100:.1f}% reduction)")


Memory usage of 'manufacturer' as object: 21.53 KB
Memory usage as category: 3.56 KB
Memory saved: 17.97 KB (0.1% reduction)


In [7]:
df['name_normalized'] = normalize_string_column(df['name'])
print("\nOriginal vs normalized name:")
for orig, norm in zip(df['name'].head(3), df['name_normalized'].head(3)):
    print(f"{orig:30} -> {norm}")


Original vs normalized name:
chevrolet chevelle malibu      -> chevrolet chevelle malibu
buick skylark 320              -> buick skylark 320
plymouth satellite             -> plymouth satellite


In [8]:
print("\n--- Final dtypes after cleaning ---")
print(df.dtypes)
print("\nMissing values summary:")
print(df.isna().sum())


--- Final dtypes after cleaning ---
mpg                       float64
cylinders                   int64
displacement              float64
horsepower                float64
weight                      int64
acceleration              float64
model_year                  int64
origin                     object
name                       object
model_date         datetime64[ns]
model_year_full             int32
name_clean                 object
manufacturer             category
name_normalized            object
dtype: object

Missing values summary:
mpg                0
cylinders          0
displacement       0
horsepower         6
weight             0
acceleration       0
model_year         0
origin             0
name               0
model_date         0
model_year_full    0
name_clean         0
manufacturer       0
name_normalized    0
dtype: int64
